# CS 3120/5120: Secure Distributed Computation
## Homework 4

In [ ]:
# Useful imports and utility functions
import pychor
import galois
import pandas as pd
import numpy as np

GF = galois.GF(2**31-1)

p1 = pychor.Party('doctor')
p2 = pychor.Party('hospital')

@pychor.local_function
def share(x):
    s1 = GF.Random()
    s2 = GF(x) - s1
    return s1, s2

This assignment covers applications in *vertically federated analytics* with two parties. In this scenario, a hospital and a doctor's office are collaborating to learn about heart disease. The hospital knows information about the heart disease status of its patients, and the doctor's office knows information about demographics and medical history of each patient (age, sex, cholesterol, blood pressure).

The goal is to calculate various statistics about the relationships between a heart disease diagnosis and other factors (age, sex, blood pressure, cholestorol, etc).

For the questions in this assignment, I recommend using the `SecInt` class from [Chapter 5 of the textbook](https://jnear.github.io/programming-mpc/chapters/chapter05.html#secint-secure-integers), with multiplication triples for multiplication and the ideal functionality for generating the triples.

The datasets are available as CSV files. The cell below loads them from the course github repository into [Pandas dataframes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html). Your code should use the functions `get_doctor_data` and `get_hospital_data` to get located versions of these datasets (do not reference `_doctor_df` or `_hospital_df` directly).

In [ ]:
_doctor_df = pd.read_csv('https://raw.githubusercontent.com/jnear/cs3120-secure-computation/refs/heads/main/data/heart_doctor.csv')
_hospital_df = pd.read_csv('https://raw.githubusercontent.com/jnear/cs3120-secure-computation/refs/heads/main/data/heart_hospital.csv')

def get_doctor_data():
    return p1.constant(_doctor_df)

def get_hospital_data():
    return p2.constant(_hospital_df)

NUM_ROWS = 303

## Question 1 (20 points)

Write a protocol to calculate the number of men (sex = 1) and women (sex = 0) who have heart disease (diagnosis = 1).

Your solution should return a tuple of two values: the number of men with heart disease, and the number of women with heart disease.

Hint: This is harder than it sounds, because the data is vertically federated! 

- I recommend secret sharing values from the appropriate columns using the `SecInt` class from Chapter 5. The definition (with supporting code) appears at the bottom of this notebook.
- You can get a list of the values from a column by converting to a Pandas `Series` then to a numpy array: `df['sex'].values`
- To count the number of men who have heart disease, you can create `SecInt` objects for the sex and diagnosis values for each row of the dataset, then compute a 1 for each man with heart disease by multiplying the value of the `sex` column by the value of the `diagnosis` column, and finally add up the values of these multiplications. 
- This will require a lot of multiplication triples! Your protocol should make sure there are enough. The `gen_triples` protocol, defined at the bottom of this notebook, can help.
- To count the number of women who have heart disease, you can use the same approach, but first "invert" the value of the `sex` column. 
- You can "invert" a field element (0 becomes 1, and 1 becomes 0) with the expression `invert(x) = -x + 1`.

In [ ]:
def men_women_diagnosis_counts():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    men_count, women_count = men_women_diagnosis_counts()
    print('Men with heart disease:', men_count)
    print('Women with heart disease:', women_count)
    assert men_count.val == 93
    assert women_count.val == 72

## Question 2 (30 points)
 
Write a protocol to produce a [histogram](https://en.wikipedia.org/wiki/Histogram) relating age to heart disease diagnosis. Your solution should count the number of people *with* and *without* heart disease in the following bins:
- age 0-39
- age 40-49
- age 50-59
- age 60-69
- age 70-120

Your solution should produce two lists:
- One list of counts for *with heart disease*
- One list of counts for *without heart disease*

Hint: It's easiest to *re-share* the `age` column for each bin, and to turn `age` into a 1/0 indicator value for inclusion in the bin via a local function *before* sharing.

In [ ]:
def age_diagnosis_histogram():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    counts_with, counts_without = age_diagnosis_histogram()
    c_with = [int(x) for x in pychor.get_val(counts_with)[0]]
    c_without = [int(x) for x in pychor.get_val(counts_without)[0]]
    print('Histogram by age of individuals with heart disease:', c_with)
    print('Histogram by age of individuals without heart disease:', c_without)
    assert c_with == [9, 47, 60, 30, 6]
    assert c_without == [156, 118, 105, 135, 159]

## Question 3 (30 points)

Write a protocol to calculate the [sample Pearson correlation coefficient](https://en.wikipedia.org/wiki/Pearson_correlation_coefficient#For_a_sample) for the `chestPain` and `diagnosis` columns. Your protocol may reveal the average and variance of both columns, as well as derived and derivable values.

In [ ]:
def chestPain_diagnosis_correlation():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    true_answer = _doctor_df['chestPain'].corr(_hospital_df['diagnosis'])
    print('True correlation coefficient:', true_answer)

    correlation = chestPain_diagnosis_correlation()
    print('Protocol correlation coefficient:', correlation)

    assert abs(correlation.val - true_answer) <= 0.001

# Useful Definitions

In [ ]:
from dataclasses import dataclass

multiplication_triples = []

@dataclass
class SecInt:
    # s1 is p1's share of the value, and s2 is p2's share
    s1: galois.GF
    s2: galois.GF

    @classmethod
    def input(cls, val):
        """Secret share an input: p1 holds s1, and p2 holds s2"""
        s1, s2 = share(val).untup(2)
        if p1 in val.parties:
            s2.send(p1, p2)
            return SecInt(s1, s2)
        else:
            s1.send(p2, p1)
            return SecInt(s1, s2)

    def __add__(x, y):
        """Add two SecInt objects using local addition of shares"""
        return SecInt(x.s1 + y.s1, x.s2 + y.s2)

    def __mul__(x, y):
        """Multiply two SecInt objects using a triple"""
        triple = multiplication_triples.pop()
        r1, r2 = protocol_mult((x.s1, x.s2), (y.s1, y.s2), triple)
        return SecInt(r1, r2)

    def reveal(self):
        """Reveal the secret value by broadcast and reconstruction"""
        self.s1.send(p1, p2)
        self.s2.send(p2, p1)
        return self.s1 + self.s2

def functionality_gen_triple():
    Fgen = pychor.Party('Fgen')

    def deal_shares(x):
        s1, s2 = share(x).untup(2)
        s1.send(Fgen, p1)
        s2.send(Fgen, p2)
        return s1, s2

    # Step 1: generate a, b, c
    a = Fgen.constant(GF.Random())
    b = Fgen.constant(GF.Random())
    c = a * b

    # Step 2: secret share a, b, c
    a1, a2 = deal_shares(a)
    b1, b2 = deal_shares(b)
    c1, c2 = deal_shares(c)
    return (a1, a2), (b1, b2), (c1, c2)

def protocol_mult(x, y, triple):
    x1, x2 = x
    y1, y2 = y
    (a1, a2), (b1, b2), (c1, c2) = triple

    # Step 1. P1 computes d_1 = x_1 - a_1 and sends the result to P2
    d1 = x1 - a1
    d1.send(p1, p2)

    # Step 2. P2 computes d_2 = x_2 - a_2 and sends the result to P1
    d2 = x2 - a2
    d2.send(p2, p1)

    # Step 3: P1 and P2 both compute $d = d_1 + d_2 = x - a$
    d = d1 + d2

    # Step 4. P1 computes e_1 = y_1 - b_1 and sends the result to P2
    e1 = y1 - b1
    e1.send(p1, p2)

    # Step 5. P2 computes e_2 = y_2 - b_2 and sends the result to P1    
    e2 = y2 - b2
    e2.send(p2, p1)

    # Step 6. P1 and P2 both compute $e = e_1 + e_2 = y - b$
    e = e1 + e2

    # Step 7. P1 computes r_1 = d*e + d*b_1 + e*a_1 + c_1
    r_1 = d * e + d * b1 + e * a1 + c1

    # Step 8. P2 computes r_2 = 0 + d*b_2 + e*a_2 + c_2
    r_2 = d * b2 + e * a2 + c2

    return r_1, r_2

def gen_triples(n):
    for _ in range(n):
        multiplication_triples.append(functionality_gen_triple())